In [ ]:
import pandas as pd
import numpy as np
import sklearn

df=pd.read_csv('https://raw.githubusercontent.com/anvarnarz/praktikum_datasets/main/usa_cars.csv',index_col=0)
df.head()

from sklearn.model_selection import train_test_split
train_set,test_set=train_test_split(df,test_size=0.2,random_state=42)

x=train_set.drop("price",axis=1)
y=train_set[["price"]].copy()


x_num=x.drop(["brand","model","title_status","color","vin","state","country","condition"],axis=1)
x_str=x[["brand","model","title_status","color","vin","state","country","condition"]].copy()


# x_num qilib ajratilgan , faqat raqamlardan iborat toplamni standard diapazonga olib kelish uchun qolib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
num_pipeline=Pipeline([("STD",StandardScaler())])

# buyerda NUM_PIPELINE nomli o'zgaruvchi ichiga berilgan toplam
# biz belgilagan funksiyalardan o'tadi va kerakli xolatga keladi
num_pipeline.fit_transform(x_num)



# str ustunlarni ham raqamlarga aylantirib olamiz
# numerik ustunlar va str ustunlarni birlashtiramiz

# ikkita massivni birlashtirish uchun quydagi kutubxoona chaqiriladi
from sklearn.compose import ColumnTransformer

# avval matnli massivni raqamga o'tkazamiz
num_attribs=list(x_num)
cat_attribs=["brand","model","title_status","color","vin","state","country","condition"]

# umumiy birlashtiruv jarayoni
full_pipeline=ColumnTransformer([
    ("num",num_pipeline,num_attribs),
    ("cat",OneHotEncoder(handle_unknown="ignore"),cat_attribs)
])

prototip=full_pipeline.fit_transform(x)

# machinelearn ga tayyor xolatga keldi
# df butunligicha bir xil diapazondagi raqamlar toplami ( massiv ) ga aylandi
prototip

# machine learn uchun (chiziqli) linear_ regression dan foydalanamiz
from sklearn.linear_model import LinearRegression
tayyorlovchi=LinearRegression()


# manashu yerda biz bergan algoritm asosida machine learn tugallanadi
# bu qatordan pastidan machine learnnni tekshiriladi xolos
model=tayyorlovchi.fit(prototip,y)

In [ ]:
# Test to‘plamni x va y ga ajratamiz
x_test = test_set.drop("price", axis=1)
y_test = test_set[["price"]].copy()

# x_test ni ham tozalaymiz: raqamlashtiramiz
x_test_prepared = full_pipeline.transform(x_test)

# Model yordamida bashorat qilamiz
y_pred = model.predict(x_test_prepared)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE (O'rtacha mutlaq xatolik):", mae)
print("MSE (O'rtacha kvadrat xatolik):", np.sqrt(mse))
print("R² (Determinatsiya koeffitsienti):", r2)

MAE (O'rtacha mutlaq xatolik): 4534.349403708169
MSE (O'rtacha kvadrat xatolik): 6770.840785767196
R² (Determinatsiya koeffitsienti): 0.710748885267737


In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, y_pred)
print("MAPE:", round(mape * 100, 2), "%")

MAPE: 1.789131360864471e+19 %


In [ ]:
# random forest algoritimida tekshirib ko'ramiz
from sklearn.ensemble import RandomForestRegressor
RN_model=RandomForestRegressor()
RN_model.fit(prototip,y)

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor()

In [ ]:
# Test to‘plamni x va y ga ajratamiz
x_test = test_set.drop("price", axis=1)
y_test = test_set[["price"]].copy()

# x_test ni ham tozalaymiz: raqamlashtiramiz
x_test_prepared = full_pipeline.transform(x_test)

# Model yordamida bashorat qilamiz
y_pred = RN_model.predict(x_test_prepared)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE (O'rtacha mutlaq xatolik):", mae)
print("MSE (O'rtacha kvadrat xatolik):", np.sqrt(mse))
print("R² (Determinatsiya koeffitsienti):", r2)

MAE (O'rtacha mutlaq xatolik): 4006.3607599999996
MSE (O'rtacha kvadrat xatolik): 6496.986010584724
R² (Determinatsiya koeffitsienti): 0.7336739172181775


In [ ]:
#  kross validation algoritimida tekshiramiz

from sklearn.model_selection import cross_val_score
crossing=cross_val_score(tayyorlovchi,prototip,y,scoring="neg_mean_squared_error",cv=5)  #cv=5 => datasetni necha marotaba bo'linishi


In [ ]:
def display(x):
  print("Scores:",x,"$")
  print("Mean:",x.mean(),"$")
  print("Standard deviation:",x.std(),"$")

In [ ]:
display(np.sqrt(-crossing))

Scores: [7061.23792421 6858.92181928 6888.98219486 7136.72672666 7955.78840553] $
Mean: 7180.33141410769 $
Standard deviation: 401.39921994417494 $


In [ ]:
import pickle
filename="ML.pkl"
with open(filename,"wb") as file:
  pickle.dump([crossing,RN_model,model],file)